# 🔍 Detecção de Fraude em Transações Financeiras
---
| Campo | Valor |
|---|---|
| **Projeto** | Detecção de Fraude com Machine Learning |
| **Autor** | `SEU NOME` |
| **Programa** | Programa de Empregabilidade |
| **Dataset** | Credit Card Fraud Detection — Kaggle |
| **Objetivo** | Construir e comparar modelos de ML para detectar transações fraudulentas |

---
## Problemática

Fraudes em cartões de crédito causam prejuízos bilionários no mundo todo. O desafio central não é técnico apenas — é estatístico: em um dataset real, menos de 0.2% das transações são fraudes. Um modelo que classifica TUDO como legítimo acerta 99.8% das vezes, mas é completamente inútil.

**A pergunta que vamos responder:** É possível construir um modelo que identifique fraudes com alta precisão sem gerar alertas falsos excessivos?

**Por que isso é difícil:**
- Dataset extremamente desbalanceado (classe minoritária = fraude)
- Custo de erro assimétrico: falso negativo (fraude não detectada) é muito pior que falso positivo
- Acurácia não serve como métrica — precisamos de Precision, Recall, F1 e AUC-ROC
- Duelo final: XGBoost vs Árvore de Decisão — qual lida melhor com desbalanceamento?

## Como baixar o dataset

1. Acesse: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
2. Crie uma conta gratuita no Kaggle
3. Baixe o arquivo `creditcard.csv` (144 MB)
4. Coloque na pasta `data/raw/`

> O dataset tem 284.807 transações reais de cartão de crédito europeu de 2013.
> 492 são fraudes (0.172%). Features V1-V28 são componentes PCA por privacidade.


## 1 · Imports e configuração

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, precision_recall_curve,
                              average_precision_score, f1_score)
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier

# Visualização
plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})

# Paths
RAW = Path('data/raw')
PROCESSED = Path('data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

print('✅ Ambiente configurado')
print('Instale se necessário: pip install imbalanced-learn xgboost')


## 2 · Carregamento e inspeção inicial

In [ ]:
df = pd.read_csv(RAW / 'creditcard.csv')
df_raw = df.copy()

print(f'Shape: {df.shape[0]:,} transações × {df.shape[1]} features')
print(f'\nMemória: {df.memory_usage().sum() / 1e6:.1f} MB')
df.head()


In [ ]:
print('=== TIPOS ===')
print(df.dtypes.to_string())
print(f'\nNulos: {df.isnull().sum().sum()}')
print(f'Duplicatas: {df.duplicated().sum()}')


In [ ]:
# Distribuição das classes — o problema central do projeto
fraudes = df['Class'].value_counts()
pct_fraude = df['Class'].mean() * 100

print('=== DISTRIBUIÇÃO DAS CLASSES ===')
print(f'Legítimas: {fraudes[0]:,} ({100-pct_fraude:.3f}%)')
print(f'Fraudes:   {fraudes[1]:,} ({pct_fraude:.3f}%)')
print(f'\nRatio de desbalanceamento: 1:{int(fraudes[0]/fraudes[1])}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fraudes.plot(kind='bar', ax=axes[0], color=['#378ADD','#E24B4A'], edgecolor='white')
axes[0].set_title('Contagem absoluta — note a escala!', fontweight='bold')
axes[0].set_xticklabels(['Legítima', 'Fraude'], rotation=0)

axes[1].pie([fraudes[0], fraudes[1]], labels=['Legítima', 'Fraude'],
            colors=['#378ADD','#E24B4A'], autopct='%1.3f%%', startangle=90)
axes[1].set_title('Proporção real das classes', fontweight='bold')
plt.tight_layout()
plt.show()

print('\n⚠️  PONTO CRÍTICO: Um modelo que classifica TUDO como legítimo')
print(f'   teria {100-pct_fraude:.2f}% de acurácia — e seria completamente inútil!')


## 3 · Análise Exploratória (EDA)

In [ ]:
# Comparar distribuição de Amount entre fraudes e legítimas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df[df['Class']==0]['Amount'].hist(bins=50, ax=axes[0], color='#378ADD',
    edgecolor='white', alpha=0.8)
axes[0].set_title('Valor das transações LEGÍTIMAS', fontweight='bold')
axes[0].set_xlabel('Valor (€)')

df[df['Class']==1]['Amount'].hist(bins=50, ax=axes[1], color='#E24B4A',
    edgecolor='white', alpha=0.8)
axes[1].set_title('Valor das transações FRAUDULENTAS', fontweight='bold')
axes[1].set_xlabel('Valor (€)')
plt.tight_layout()
plt.show()

print('Legítimas — mediana:', df[df['Class']==0]['Amount'].median())
print('Fraudes   — mediana:', df[df['Class']==1]['Amount'].median())


In [ ]:
# Distribuição temporal: fraudes por hora do dia
df['Hour'] = (df['Time'] / 3600) % 24

fig, ax = plt.subplots(figsize=(12, 4))
df[df['Class']==0].groupby(df['Hour'].astype(int))['Class'].count().plot(
    ax=ax, color='#378ADD', label='Legítimas', alpha=0.7)
df[df['Class']==1].groupby(df['Hour'].astype(int))['Class'].count().plot(
    ax=ax, color='#E24B4A', label='Fraudes', alpha=0.9)
ax.set_title('Volume de transações por hora do dia', fontweight='bold')
ax.set_xlabel('Hora')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Features com maior diferença entre classes — pistas para o modelo
from scipy import stats

print('Features com maior separação entre classes (teste t):')
resultados = []
for col in [c for c in df.columns if c.startswith('V')]:
    leg = df[df['Class']==0][col]
    fra = df[df['Class']==1][col]
    stat, p = stats.ttest_ind(leg, fra)
    diff_media = abs(leg.mean() - fra.mean())
    resultados.append({'feature': col, 'diff_media': diff_media, 'p_value': p})

res_df = pd.DataFrame(resultados).sort_values('diff_media', ascending=False)
print(res_df.head(10).to_string(index=False))
print('\nAs top features são as mais promissoras para o modelo!')


In [ ]:
# Correlação das top features com a classe
top_features = res_df.head(10)['feature'].tolist()
corr_class = df[top_features + ['Class']].corr()['Class'].drop('Class').sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
cores = ['#E24B4A' if v > 0 else '#378ADD' for v in corr_class]
corr_class.plot(kind='barh', ax=ax, color=cores, edgecolor='white')
ax.set_title('Correlação das top features com fraude (vermelho = positiva)', fontweight='bold')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()


## 4 · Pré-processamento e tratamento do desbalanceamento

In [ ]:
# Normalizar Amount e Time (V1-V28 já estão em escala PCA)
df['Amount_scaled'] = StandardScaler().fit_transform(df[['Amount']])
df['Hour_scaled'] = StandardScaler().fit_transform(df[['Hour']])
df = df.drop(['Time', 'Amount'], axis=1)

# Separar features e target
feature_cols = [c for c in df.columns if c != 'Class']
X = df[feature_cols]
y = df['Class']

# Split estratificado — mantém proporção de fraudes em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Treino: {len(X_train):,} amostras | Fraudes: {y_train.sum()} ({y_train.mean()*100:.2f}%)')
print(f'Teste:  {len(X_test):,} amostras  | Fraudes: {y_test.sum()} ({y_test.mean()*100:.2f}%)')


In [ ]:
# SMOTE — Synthetic Minority Oversampling Technique
# Cria exemplos sintéticos de fraudes para balancear as classes
# IMPORTANTE: aplica SOMENTE no treino, nunca no teste!

print('Antes do SMOTE:')
print(f'  Legítimas: {(y_train==0).sum():,}')
print(f'  Fraudes:   {(y_train==1).sum():,}')

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('\nDepois do SMOTE:')
print(f'  Legítimas: {(y_train_sm==0).sum():,}')
print(f'  Fraudes:   {(y_train_sm==1).sum():,}')
print('\n⚠️  SMOTE só é aplicado no treino — o teste permanece com a distribuição REAL!')


## 5 · Modelagem — Árvore de Decisão

In [ ]:
# Árvore de Decisão com class_weight para lidar com desbalanceamento
dt = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42
)

dt.fit(X_train_sm, y_train_sm)
y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

print('=== ÁRVORE DE DECISÃO ===')
print(classification_report(y_test, y_pred_dt, target_names=['Legítima','Fraude']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_dt):.4f}')
print(f'Average Precision: {average_precision_score(y_test, y_prob_dt):.4f}')


In [ ]:
# Visualizar matriz de confusão da Árvore
cm_dt = confusion_matrix(y_test, y_pred_dt)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_dt, annot=True, fmt=',', cmap='Blues', ax=ax,
            xticklabels=['Legítima','Fraude'], yticklabels=['Legítima','Fraude'])
ax.set_title('Matriz de Confusão — Árvore de Decisão', fontweight='bold')
ax.set_ylabel('Real')
ax.set_xlabel('Previsto')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm_dt.ravel()
print(f'\nVerdadeiros Negativos (legítimas corretas): {tn:,}')
print(f'Falsos Positivos (legítimas marcadas como fraude): {fp:,}')
print(f'Falsos Negativos (fraudes NÃO detectadas): {fn}  ← crítico!')
print(f'Verdadeiros Positivos (fraudes detectadas): {tp}')


## 6 · Modelagem — XGBoost

In [ ]:
# Calcular scale_pos_weight para lidar com desbalanceamento nativamente
scale = (y_train==0).sum() / (y_train==1).sum()
print(f'scale_pos_weight: {scale:.1f}')

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale,  # lida nativamente com desbalanceamento
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='aucpr',     # optimiza Average Precision (melhor para dados desbalanceados)
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train,  # XGBoost com dados originais (sem SMOTE)
        eval_set=[(X_test, y_test)], verbose=False)

y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

print('\n=== XGBOOST ===')
print(classification_report(y_test, y_pred_xgb, target_names=['Legítima','Fraude']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_xgb):.4f}')
print(f'Average Precision: {average_precision_score(y_test, y_prob_xgb):.4f}')


In [ ]:
# Feature importance do XGBoost
feat_imp = pd.Series(xgb.feature_importances_, index=feature_cols)
feat_imp = feat_imp.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
feat_imp.plot(kind='bar', ax=ax, color='#378ADD', edgecolor='white')
ax.set_title('Top 15 features mais importantes — XGBoost', fontweight='bold')
ax.set_xlabel('')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7 · Duelo: XGBoost vs Árvore de Decisão

In [ ]:
# Comparação completa das métricas
metricas = {
    'Modelo': ['Árvore de Decisão', 'XGBoost'],
    'AUC-ROC': [
        roc_auc_score(y_test, y_prob_dt),
        roc_auc_score(y_test, y_prob_xgb)
    ],
    'Avg Precision': [
        average_precision_score(y_test, y_prob_dt),
        average_precision_score(y_test, y_prob_xgb)
    ],
    'F1 (Fraude)': [
        f1_score(y_test, y_pred_dt, pos_label=1),
        f1_score(y_test, y_pred_xgb, pos_label=1)
    ],
    'Recall (Fraude)': [
        confusion_matrix(y_test, y_pred_dt).ravel()[3] / y_test.sum(),
        confusion_matrix(y_test, y_pred_xgb).ravel()[3] / y_test.sum()
    ]
}

df_metricas = pd.DataFrame(metricas).set_index('Modelo').round(4)
print('=== DUELO FINAL ===')
print(df_metricas.to_string())
vencedor = df_metricas['AUC-ROC'].idxmax()
print(f'\n🏆 Vencedor por AUC-ROC: {vencedor}')


In [ ]:
# Curvas ROC lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva ROC
for prob, nome, cor in [(y_prob_dt,'Árvore','#378ADD'),(y_prob_xgb,'XGBoost','#E24B4A')]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    axes[0].plot(fpr, tpr, color=cor, lw=2, label=f'{nome} (AUC={auc:.4f})')
axes[0].plot([0,1],[0,1],'k--',lw=1)
axes[0].set_title('Curva ROC — XGBoost vs Árvore', fontweight='bold')
axes[0].set_xlabel('Taxa de Falso Positivo')
axes[0].set_ylabel('Taxa de Verdadeiro Positivo')
axes[0].legend()

# Curva Precision-Recall (mais informativa para dados desbalanceados)
for prob, nome, cor in [(y_prob_dt,'Árvore','#378ADD'),(y_prob_xgb,'XGBoost','#E24B4A')]:
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    axes[1].plot(rec, prec, color=cor, lw=2, label=f'{nome} (AP={ap:.4f})')
axes[1].set_title('Curva Precision-Recall ← mais importante aqui!', fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

plt.tight_layout()
plt.show()

print('\n💡 A curva Precision-Recall é mais informativa que ROC para dados desbalanceados!')


## 8 · Conclusões e análise crítica

### Principais achados

1. **[Preencha após rodar]** — XGBoost obteve AUC-ROC de X vs Y da Árvore
2. **Desbalanceamento importa** — sem tratamento, os modelos teriam performance enganosa
3. **Métricas corretas salvam projetos** — acurácia de 99%+ seria mentira estatística

### Por que XGBoost performa melhor (provavelmente)
- Ensemble de centenas de árvores corrigindo erros iterativamente
- `scale_pos_weight` penaliza mais os erros na classe minoritária
- Regularização L1/L2 evita overfitting
- Subsampling de features e linhas aumenta robustez

### Limitações desta análise
- Features V1-V28 são PCA anonimizado — não sabemos o significado real
- Dataset de 2013 — padrões de fraude mudaram
- Threshold padrão de 0.5 pode não ser ótimo — ajustar conforme custo do negócio

### Próximos passos
- Otimizar threshold de decisão com base no custo de cada tipo de erro
- Testar outras técnicas de balanceamento (ADASYN, Borderline-SMOTE)
- Adicionar Random Forest como terceiro competidor
- Implementar explicabilidade com SHAP values


In [ ]:
# Salvar dados processados e resultados
df_metricas.to_csv(PROCESSED / 'metricas_comparacao.csv')
print('✅ Resultados salvos em data/processed/')
print(f'   Shape final dataset: {df.shape}')
